# 1장. Custom Middleware — Node-style: @before_model + can_jump_to

**Node-style Hook**은 특정 실행 지점에서 순차적으로 동작하는 미들웨어입니다.

- `@before_agent` : Agent 전체 실행에서 **딱 한 번** 실행
- `@before_model` : **Model이 호출될 때마다** 실행 (Tool 결과를 받고 재호출할 때도 포함)

이번 예시에서는 `@before_model(can_jump_to=["end"])`를 사용해 사용자 입력에 "암구호"가 포함된 경우  
LLM 호출 없이 즉시 종료하는 **입력 필터 가드레일**을 구현합니다.

> `can_jump_to=["end"]` — 이 미들웨어가 `"end"` 노드로 바로 점프할 수 있는 권한 선언  
> `return None` — 점프 없이 평소대로 다음 단계 진행  
> `return {"jump_to": "end"}` — LLM 호출 없이 Agent 즉시 종료

In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent


# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-4o-mini")

In [4]:
from langchain.agents.middleware import before_model
from langchain.messages import AIMessage

@before_model(can_jump_to=["end"])
def validate_input(state, runtime):
    last_message = state["messages"][-1]
    if "암구호" in last_message.content:
        print("암구호 감지됨 — 응답 차단")
        return {
            "messages": [AIMessage(content="이 요청은 처리할 수 없습니다.")],
            "jump_to": "end"  # 모델 호출 중단 후 에이전트 종료
        }
    print("✅ 정상 입력, 모델 호출 계속 진행")
    return None

## 실행 결과 분석

"암구호" 감지 시 `messages` 리스트에는 `HumanMessage`와 차단 `AIMessage`만 쌓이고,  
**실제 LLM 호출은 일어나지 않습니다.** → AI Model에게 민감 정보가 유출되는 것을 사전에 방지합니다.

In [5]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

agent = create_agent(
    model=init_chat_model("gpt-5-nano"),
    tools=[],
    middleware=[validate_input],
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "오늘의 암구호는 삼각대-자동차 입니다."}]},
)
print(response)


암구호 감지됨 — 응답 차단
{'messages': [HumanMessage(content='오늘의 암구호는 삼각대-자동차 입니다.', additional_kwargs={}, response_metadata={}, id='c2349713-c8fa-408d-b511-1daafe796cab'), AIMessage(content='이 요청은 처리할 수 없습니다.', additional_kwargs={}, response_metadata={}, id='ba386814-bac9-4740-8162-7dd9d5f45a87', tool_calls=[], invalid_tool_calls=[])]}
